In [0]:
%sql
SELECT 
    da.artist_name,
    ROUND(AVG(ft.popularity), 2) AS avg_popularity
FROM dev_workspace.gold.fact_tracks ft
JOIN dev_workspace.gold.dim_artist da 
    ON ft.artist_id = da.artist_id
GROUP BY da.artist_name
HAVING AVG(ft.popularity) > (
    SELECT AVG(popularity) 
    FROM dev_workspace.gold.fact_tracks
)
order by avg_popularity desc
limit 10;

In [0]:
%sql
SELECT 
    ft.track_name,
    dg.genre_name,
    ft.duration_ms
FROM dev_workspace.gold.fact_tracks ft
JOIN dev_workspace.gold.dim_genre dg 
    ON ft.genre_id = dg.genre_id
WHERE ft.duration_ms > (
    SELECT AVG(duration_ms)
    FROM dev_workspace.gold.fact_tracks f2
    WHERE f2.genre_id = ft.genre_id
)
order by ft.duration_ms desc
limit 10;

In [0]:
%sql
SELECT artist_name, explicit_count
FROM (
    SELECT 
        da.artist_name,
        COUNT(*) AS explicit_count
    FROM dev_workspace.gold.fact_tracks ft
    JOIN dev_workspace.gold.dim_artist da 
        ON ft.artist_id = da.artist_id
    WHERE ft.explicit = TRUE
    GROUP BY da.artist_name
) sub
ORDER BY explicit_count DESC
LIMIT 5;

In [0]:
%sql
SELECT 
    dal.album_name,
    da.artist_name,
    COUNT(ft.track_id) AS tracks_in_album
FROM dev_workspace.gold.fact_tracks ft
JOIN dev_workspace.gold.dim_album dal 
    ON ft.album_id = dal.album_id
JOIN dev_workspace.gold.dim_artist da 
    ON dal.artist_id = da.artist_id
GROUP BY dal.album_name, da.artist_name
HAVING COUNT(ft.track_id) > (
    SELECT AVG(album_size)
    FROM (
        SELECT COUNT(track_id) AS album_size
        FROM dev_workspace.gold.fact_tracks
        GROUP BY album_id
    ) sub_album
)
ORDER BY tracks_in_album DESC
LIMIT 10;

In [0]:
%sql
SELECT 
    da.artist_name,
    ft.track_name,
    ROUND(daf.danceability, 3) AS danceability
FROM dev_workspace.gold.fact_tracks ft
JOIN dev_workspace.gold.dim_artist da 
    ON ft.artist_id = da.artist_id
JOIN dev_workspace.gold.dim_audio_features daf 
    ON ft.audio_id = daf.audio_id
WHERE daf.danceability > (
    SELECT AVG(danceability) 
    FROM dev_workspace.gold.dim_audio_features
)
ORDER BY danceability DESC
LIMIT 20;

In [0]:
%sql
SELECT 
    dg.genre_name,
    COUNT(DISTINCT ft.track_id) AS total_tracks,       -- cuántas canciones hay por género
    ROUND(AVG(ft.popularity), 2) AS avg_popularity     -- popularidad promedio del género
FROM dev_workspace.gold.fact_tracks ft
JOIN dev_workspace.gold.dim_genre dg 
    ON ft.genre_id = dg.genre_id
GROUP BY dg.genre_name
ORDER BY total_tracks DESC, avg_popularity DESC
LIMIT 10;